# R3 — Balanceo, solo sobre train

## Qué corrige

En la v1 (notebooks 2 y 6) el remuestreo se aplicaba al dataset **completo**,
antes de separar el holdout. Con *random oversampling* eso duplicaba
literalmente las sesiones de vishing del conjunto de evaluación dentro del
entrenamiento: medimos **500/500 (100 %)** en la configuración del mejor modelo
del manuscrito.

Ese es el origen del hallazgo del paper de que el *random oversampling* supera
a SMOTE. RO copia los positivos del test; SMOTE solo interpola. La ventaja era
un artefacto de fuga.

Aquí el remuestreo se aplica **exclusivamente a train**, y cada dataset
resultante se somete a la aserción anti-fuga contra val y test.

In [1]:
# Celda de arranque idéntica en todos los notebooks. Deja el kernel de SageMaker
# en un estado conocido: mismo directorio de trabajo, mismas semillas, mismas
# versiones. Si algo de esto cambia entre corridas, los resultados no son
# comparables aunque los notebooks sean los mismos.
import sys, os

AQUI = os.getcwd()                    # los notebooks y vishing_common.py conviven
if AQUI not in sys.path:
    sys.path.insert(0, AQUI)

import numpy as np
import pandas as pd
import vishing_common as vc

vc.set_all_seeds()                    # random, numpy y torch (+ cuDNN determinista)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("dataset:", vc.RAW_FILENAME, "(esquema", vc.DATASET_VERSION + ")")
print("bucket :", vc.BUCKET, "| prefijo:", vc.PREFIX)
print("seed   :", vc.SEED, "| split:", vc.SPLIT_MODE, "| política:", vc.FEATURE_POLICY)
print("xgboost se ejecutará en:", vc.xgb_device())
vc.check_versions()

dataset: biocatch_sinthetic_data_v3.csv (esquema v3)
bucket : poc-vishing | prefijo: v2
seed   : 42 | split: grouped | política: audited
xgboost se ejecutará en: cuda
  versiones OK: numpy 2.0.2, pandas 2.2.3, scipy 1.14.1, sklearn 1.5.2, imblearn 0.12.4, xgboost 2.1.4


,paquete,instalada,esperado,ok
0,numpy,2.0.2,">=1.26,<2.1",True
1,pandas,2.2.3,">=2.1,<2.3",True
2,scipy,1.14.1,">=1.11,<1.15",True
3,sklearn,1.5.2,">=1.4,<1.6",True
4,imblearn,0.12.4,">=0.12,<0.13",True
5,xgboost,2.1.4,">=2.0,<2.2",True


In [2]:
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

cfg = vc.read_json(vc.P.feature_contract)
contract = cfg["contratos"][cfg["activo"]]

val  = vc.read_parquet(vc.P.val)
test = vc.read_parquet(vc.P.test)

CANDIDATAS = {
    "original":  vc.P.train,
    "augmented": vc.P.train_augmented,
}

# Si R2 no se ha ejecutado, `train_augmented` no existe. Antes eso reventaba a
# mitad del bucle con un error de S3 poco informativo; ahora se omite la fuente
# y se avisa, de modo que R3 pueda correrse solo con datos originales.
FUENTES = {}
for nombre, ruta in CANDIDATAS.items():
    try:
        vc.read_parquet(ruta).head(1)
        FUENTES[nombre] = ruta
    except Exception as e:
        print("AVISO: se omite la fuente '%s' (%s no disponible): %s"
              % (nombre, ruta, type(e).__name__))
assert FUENTES, "no hay ninguna fuente de train disponible; ejecuta R0 primero"

RATIOS = [0.10, 0.20, 0.25]
print("fuentes:", list(FUENTES))
print("features:", contract["n_features"], "| ratios:", RATIOS)

fuentes: ['original', 'augmented']
features: 44 | ratios: [0.1, 0.2, 0.25]


In [3]:
def tecnicas(n_may):
    def estrategia(pct):
        return pct / (1.0 - pct)
    return {
        "random_oversampling": lambda p: RandomOverSampler(
            sampling_strategy=estrategia(p), random_state=vc.SEED),
        "smote": lambda p: SMOTE(
            sampling_strategy=estrategia(p), random_state=vc.SEED),
        "borderline_smote": lambda p: BorderlineSMOTE(
            sampling_strategy=estrategia(p), random_state=vc.SEED, kind="borderline-1"),
        "smote_undersampling": lambda p: Pipeline([
            ("o", SMOTE(sampling_strategy={1: int(int(n_may * 0.9) * (p / (1 - p)))},
                        random_state=vc.SEED)),
            ("u", RandomUnderSampler(sampling_strategy={0: int(n_may * 0.9)},
                                     random_state=vc.SEED)),
        ]),
    }

## Remuestreo

Punto clave: `row_id` y `origin` **no** entran al remuestreo como features,
pero sí se reconstruyen después. Para las técnicas que sintetizan filas nuevas
(SMOTE y variantes) las filas generadas reciben `row_id = -1`: no corresponden
a ninguna sesión real, y marcarlas así evita que una aserción de identidad las
dé por buenas.

Segundo punto clave, nuevo con el esquema v3: **SMOTE interpola sobre el
centinela**. Una fila sintética entre una sesión con transacción de 3.000.000
COP y otra sin transacción (`transaction_amount_cop = -1`) sale con un monto
intermedio y `transaction_attempted = 0,5` → redondeado a 0. Es una combinación
que no existe en el dataset real. Por eso, tras cada remuestreo se aplica
`vc.repair_coherence`, la misma función que usa R2 sobre las filas de CTGAN, y
se verifica con `vc.check_coherence` que no queda ninguna violación. El
`random_oversampling` no lo necesita (copia filas enteras), pero se le aplica
igual para que todos los datasets pasen por el mismo camino.

In [4]:
resumen = []

for data_type, ruta in FUENTES.items():
    tr = vc.read_parquet(ruta)
    X = tr[contract["features"]].fillna(0)
    y = tr[vc.TARGET]
    ids = tr[vc.ROW_ID].values
    n_may = int((y == 0).sum())
    print("\n===== %s : %d filas, vishing %.4f =====" % (data_type, len(tr), y.mean()))

    # -- baseline sin balanceo --
    # La v1 lo incluía como decimotercer dataset. Es el control necesario para
    # saber si el remuestreo aporta algo o si el desbalance se maneja mejor con
    # scale_pos_weight, y R4 lo recorre igual que a los demás.
    base = tr[contract["features"] + [vc.ROW_ID, vc.TARGET, vc.ORIGIN]].copy()
    vc.assert_disjoint(base[base[vc.ROW_ID] >= 0], val, "%s/none vs val" % data_type)
    vc.assert_disjoint(base[base[vc.ROW_ID] >= 0], test, "%s/none vs test" % data_type)
    vc.write_parquet(base, vc.P.balanced(data_type, "none", 0))
    resumen.append({"data_type": data_type, "tecnica": "none", "ratio": 0,
                    "filas": len(base), "sinteticas": 0,
                    "tasa_vishing": round(float(base[vc.TARGET].mean()), 4)})

    for nombre, hacer in tecnicas(n_may).items():
        for pct in RATIOS:
            sampler = hacer(pct)
            # Se anexa row_id como columna extra para poder rastrear el origen
            # de cada fila remuestreada; se retira antes de guardar las features.
            Xi = X.copy()
            Xi["__row_id"] = ids
            Xr, yr = sampler.fit_resample(Xi, y)

            out = pd.DataFrame(Xr, columns=Xi.columns)
            out[vc.ROW_ID] = out.pop("__row_id").round().astype(np.int64)
            out[vc.TARGET] = np.asarray(yr)

            # Las filas sintetizadas por SMOTE no corresponden a sesiones reales
            reales = set(ids.tolist())
            out.loc[~out[vc.ROW_ID].isin(reales), vc.ROW_ID] = -1
            out[vc.ORIGIN] = np.where(out[vc.ROW_ID] == -1, "resampled", "original")

            # Esquema v3: deshacer las interpolaciones imposibles de SMOTE
            n_sint = int((out[vc.ROW_ID] == -1).sum())
            if n_sint:
                out = vc.repair_coherence(out)
                viol = vc.check_coherence(out, "%s/%s/%d" % (data_type, nombre, pct * 100))
                assert sum(viol.values()) == 0, "quedan violaciones de esquema"

            vc.assert_disjoint(out[out[vc.ROW_ID] >= 0], val,
                               "%s/%s/%d vs val" % (data_type, nombre, pct * 100))
            vc.assert_disjoint(out[out[vc.ROW_ID] >= 0], test,
                               "%s/%s/%d vs test" % (data_type, nombre, pct * 100))

            uri = vc.P.balanced(data_type, nombre, int(pct * 100))
            vc.write_parquet(out, uri)
            resumen.append({"data_type": data_type, "tecnica": nombre,
                            "ratio": int(pct * 100), "filas": len(out),
                            "sinteticas": n_sint,
                            "tasa_vishing": round(float(out[vc.TARGET].mean()), 4)})

display(pd.DataFrame(resumen))


===== original : 60134 filas, vishing 0.0500 =====
  OK sin fuga [original/none vs val]: 60,134 train / 19,673 eval, disjuntos
  OK sin fuga [original/none vs test]: 60,134 train / 20,193 eval, disjuntos
  escrito s3://poc-vishing/v2/04_balanced/original/none/0.parquet  (60,134 filas x 47 cols)
  OK sin fuga [original/random_oversampling/10 vs val]: 60,134 train / 19,673 eval, disjuntos
  OK sin fuga [original/random_oversampling/10 vs test]: 60,134 train / 20,193 eval, disjuntos
  escrito s3://poc-vishing/v2/04_balanced/original/random_oversampling/10.parquet  (63,472 filas x 47 cols)
  OK sin fuga [original/random_oversampling/20 vs val]: 60,134 train / 19,673 eval, disjuntos
  OK sin fuga [original/random_oversampling/20 vs test]: 60,134 train / 20,193 eval, disjuntos
  escrito s3://poc-vishing/v2/04_balanced/original/random_oversampling/20.parquet  (71,406 filas x 47 cols)
  OK sin fuga [original/random_oversampling/25 vs val]: 60,134 train / 19,673 eval, disjuntos
  OK sin fuga [

,data_type,tecnica,ratio,filas,sinteticas,tasa_vishing
0,original,none,0,60134,0,0.0500
1,original,random_oversampling,10,63472,0,0.1000
2,original,random_oversampling,20,71406,0,0.2000
3,original,random_oversampling,25,76166,0,0.2500
4,original,smote,10,63472,1284,0.1000
5,original,smote,20,71406,4490,0.2000
6,original,smote,25,76166,6364,0.2500
7,original,borderline_smote,10,63472,1362,0.1000
8,original,borderline_smote,20,71406,4374,0.2000
9,original,borderline_smote,25,76166,6268,0.2500


In [5]:
print("R3 completo:", len(resumen), "datasets balanceados.")
print("Todos verificados disjuntos de val y test. Continuar con R4.")

R3 completo: 26 datasets balanceados.
Todos verificados disjuntos de val y test. Continuar con R4.
